# Data Quality and Exploratory Data Analysis

This notebook performs an initial inspection of the checkout A/B test dataset before statistical hypothesis testing.

The focus is data structure, completeness, uniqueness, category distributions, descriptive statistics, date coverage, and logical funnel consistency.

## 1. Import libraries

`pandas` handles tabular inspection, while `pathlib` makes the dataset path portable within this project.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.3f}".format)

## 2. Load the dataset


In [2]:
DATA_PATH = Path("../data/raw/ab_checkout_experiment_200k.csv")
df = pd.read_csv(DATA_PATH)
df["experiment_date"] = pd.to_datetime(df["experiment_date"], errors="coerce")

df.head()

,user_id,experiment_date,experiment_day,variant,device,country,traffic_source,new_user,sessions,checkout_started,purchase,revenue,payment_failed,refund,page_load_time_ms,exposure_logged
0,100001,2026-02-08,35,A,Desktop,AU,Organic,True,1,True,False,0.000,False,False,2536.900,True
1,100002,2026-01-07,3,B,Desktop,US,Organic,True,1,False,False,0.000,False,False,2724.500,True
2,100003,2026-02-09,36,A,Mobile,AU,Organic,True,1,True,False,0.000,False,False,2746.700,True
3,100004,2026-01-14,10,B,Mobile,CA,Paid Search,True,3,True,True,31.390,False,False,2723.900,True
4,100005,2026-01-09,5,B,Mobile,UK,Organic,True,9,False,False,0.000,False,False,2540.000,True


## 3. Basic dataset overview

Shape, column names, and data types establish the dataset grain and confirm that the fields needed for the experiment are present with usable representations.

In [3]:
overview = pd.DataFrame(
    {
        "measure": ["rows", "columns"],
        "value": [df.shape[0], df.shape[1]],
    }
)
display(overview)

display(pd.DataFrame({"column_name": df.columns, "data_type": df.dtypes.astype(str)}))

,measure,value
0,rows,200000
1,columns,16


,column_name,data_type
user_id,user_id,int64
experiment_date,experiment_date,datetime64[us]
experiment_day,experiment_day,int64
variant,variant,str
device,device,str
country,country,str
traffic_source,traffic_source,str
new_user,new_user,bool
sessions,sessions,int64
checkout_started,checkout_started,bool


## 4. Missing-value analysis


In [4]:
missing_values = df.isna().sum().rename("missing_count").to_frame()
missing_values["missing_percentage"] = (missing_values["missing_count"] / len(df) * 100).round(2)
display(missing_values)

,missing_count,missing_percentage
user_id,0,0.000
experiment_date,0,0.000
experiment_day,0,0.000
variant,0,0.000
device,0,0.000
country,0,0.000
traffic_source,0,0.000
new_user,0,0.000
sessions,0,0.000
checkout_started,0,0.000


## 5. Duplicate analysis



In [5]:
duplicate_analysis = pd.DataFrame(
    {
        "check": ["duplicate rows", "rows after first repeated user_id", "duplicated user_id values"],
        "count": [
            int(df.duplicated().sum()),
            int(df["user_id"].duplicated().sum()),
            int(df.loc[df["user_id"].duplicated(keep=False), "user_id"].nunique()),
        ],
    }
)
display(duplicate_analysis)

assert df["user_id"].is_unique, "Expected one row per user_id for this experiment dataset."

,check,count
0,duplicate rows,0
1,rows after first repeated user_id,0
2,duplicated user_id values,0


## 6. Unique values and distributions of categorical variables



In [6]:
categorical_columns = [
    "variant",
    "device",
    "country",
    "traffic_source",
    "new_user",
    "checkout_started",
    "purchase",
    "payment_failed",
    "refund",
    "exposure_logged",
]

for column in categorical_columns:
    distribution = (
        df[column]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )
    distribution["percentage"] = (distribution["count"] / len(df) * 100).round(2)
    print(f"{column}: unique values = {df[column].nunique(dropna=False)}")
    display(distribution)

variant: unique values = 2


,count,percentage
variant,,
A,100304,50.150
B,99696,49.850


device: unique values = 3


,count,percentage
device,,
Mobile,122914,61.460
Desktop,67066,33.530
Tablet,10020,5.010


country: unique values = 7


...

,count,percentage
payment_failed,,
False,198720,99.360
True,1280,0.640


refund: unique values = 2


,count,percentage
refund,,
False,198920,99.460
True,1080,0.540


exposure_logged: unique values = 2


,count,percentage
exposure_logged,,
True,199304,99.650
False,696,0.350


## 7. Descriptive statistics for numerical variables


In [7]:
numerical_columns = df.select_dtypes(include="number").columns.drop("user_id")
display(df[numerical_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
experiment_day,200000.000,21.812,11.968,1.000,12.000,22.000,32.000,42.000
sessions,200000.000,3.493,2.523,1.000,2.000,3.000,5.000,35.000
revenue,200000.000,8.248,27.326,0.000,0.000,0.000,0.000,396.220
page_load_time_ms,200000.000,2805.606,318.918,2062.700,2580.100,2763.800,2987.700,5742.800


## 8. Experiment date range



In [8]:
date_range = pd.DataFrame(
    {
        "minimum_date": [df["experiment_date"].min().date()],
        "maximum_date": [df["experiment_date"].max().date()],
        "date_count": [df["experiment_date"].nunique()],
    }
)
display(date_range)

assert df["experiment_date"].notna().all(), "Experiment dates must parse successfully."

,minimum_date,maximum_date,date_count
0,2026-01-05,2026-02-15,42


## 9. Logical consistency checks

These checks verify that the checkout funnel does not contain impossible event combinations. Violations should be investigated before calculating conversion or revenue metrics because they can make outcomes internally inconsistent.

In [9]:
logical_checks = {
    "purchase without checkout_started": df["purchase"] & ~df["checkout_started"],
    "nonzero revenue without purchase": (df["purchase"] == False) & (df["revenue"] != 0),
    "refund without purchase": df["refund"] & ~df["purchase"],
    "payment_failed without checkout_started": df["payment_failed"] & ~df["checkout_started"],
}

logical_check_results = pd.DataFrame(
    {
        "check": logical_checks.keys(),
        "violations": [int(mask.sum()) for mask in logical_checks.values()],
    }
)
display(logical_check_results)

assert logical_check_results["violations"].eq(0).all(), "Logical funnel consistency checks failed."

,check,violations
0,purchase without checkout_started,0
1,nonzero revenue without purchase,0
2,refund without purchase,0
3,payment_failed without checkout_started,0


## 10. Variant allocation

Assignment counts and percentages describe how many observations are available in each arm. 

In [10]:
variant_allocation = df["variant"].value_counts(dropna=False).rename("count").to_frame()
variant_allocation["percentage"] = (variant_allocation["count"] / len(df) * 100).round(2)
display(variant_allocation)

,count,percentage
variant,,
A,100304,50.150
B,99696,49.850


## 11. Exposure logging


In [11]:
exposure_distribution = df["exposure_logged"].value_counts(dropna=False).rename("count").to_frame()
exposure_distribution["percentage"] = (exposure_distribution["count"] / len(df) * 100).round(2)
display(exposure_distribution)

,count,percentage
exposure_logged,,
True,199304,99.650
False,696,0.350


## 12. Concise data-quality summary


In [13]:
quality_summary = pd.DataFrame(
    {
        "check": [
            "missing cells",
            "duplicate rows",
            "duplicate user_id values",
            "logical funnel violations",
            "parsed experiment dates",
        ],
        "result": [
            int(df.isna().sum().sum()),
            int(df.duplicated().sum()),
            int(df["user_id"].duplicated().sum()),
            int(logical_check_results["violations"].sum()),
            int(df["experiment_date"].notna().sum()),
        ],
        "status": ["PASS", "PASS", "PASS", "PASS", "PASS"],
    }
)
display(quality_summary)

print(
    "The dataset is structurally complete for initial analysis: "
    "no missing cells, duplicate rows, duplicate users, or funnel violations were found."
)


,check,result,status
0,missing cells,0,PASS
1,duplicate rows,0,PASS
2,duplicate user_id values,0,PASS
3,logical funnel violations,0,PASS
4,parsed experiment dates,200000,PASS


The dataset is structurally complete for initial analysis: no missing cells, duplicate rows, duplicate users, or funnel violations were found.
